In [1]:
import torch
print("device_count:", torch.cuda.device_count())
print("is_available:", torch.cuda.is_available())

device_count: 1
is_available: True


In [2]:
import sys, site

user_site = site.getusersitepackages()
print("Using python:", sys.executable)
print("User site on path:", user_site in sys.path)

Using python: /opt/conda/envs/cse234/bin/python
User site on path: True


In [3]:
import sys, site

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
from datasets import Dataset
import itertools
import json

print("RFLoraConfig:", RFLoraConfig)


RFLoraConfig: <class 'rapidfireai.automl.model_config.RFLoraConfig'>


In [4]:
import sys, site
sys.path.insert(0, site.getusersitepackages())
from peft import LoraConfig
print("peft works:", LoraConfig)

peft works: <class 'peft.tuners.lora.config.LoraConfig'>


#### Define Model Creation Function for All Model Types Across Configs

In [5]:
# from rapidfireai import Experiment
# from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
# from datasets import Dataset
# import itertools
# import json

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 512

### Load Dataset and Specify Train and Eval Partitions

In [7]:
import json
from datasets import Dataset

with open("train.json", "r") as f:
    train_dataset = Dataset.from_list(json.load(f))

with open("validation.json", "r") as f:
    validation_dataset = Dataset.from_list(json.load(f))

# print(len(train_dataset), len(validation_dataset))


### Define Data Processing Function

In [8]:

def basic_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
    "text": (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
}

def pkfk_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    column_info = schema_data["column_names_original"]
    primary_keys = schema_data.get("primary_keys", [])
    foreign_keys = schema_data.get("foreign_keys", [])
    col_annotations = {}

    for pk in primary_keys:
        if isinstance(pk, list):
            for col_idx in pk:
                col_annotations[col_idx] = "(PK)"
        else:
            col_annotations[pk] = "(PK)"

    for from_idx, to_idx in foreign_keys:
        to_table_idx = column_info[to_idx][0]
        to_table_name = schema_data["table_names_original"][to_table_idx]
        if from_idx in col_annotations and col_annotations[from_idx] == "(PK)":
            col_annotations[from_idx] = f"(PK,FK→{to_table_name})"
        else:
            col_annotations[from_idx] = f"(FK→{to_table_name})"

    schema = {}
    for col_idx, (table_idx, col_name) in enumerate(column_info):
        if table_idx == -1:
            continue
        table_name = schema_data["table_names_original"][table_idx]
        if table_name not in schema:
            schema[table_name] = []
        annotation = col_annotations.get(col_idx, "")
        if annotation:
            annotated_col = f"{col_name} {annotation}"
        else:
            annotated_col = col_name
        schema[table_name].append(annotated_col)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema with PK/FK annotations, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists (without annotations in the output)."
    )
    prompt = (
        f"Database schema (PK/FK annotated): {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only — column names without annotations: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
    "text": (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
}
def sorted_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)

    sorted_schema = {}
    for table_name in sorted(schema.keys()):
        sorted_schema[table_name] = sorted(schema[table_name])

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema (sorted): {sorted_schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
    "text": (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
}

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

def pretokenize(fmt_func):
    def _inner(row):
        text = fmt_func(row)["text"] + tokenizer.eos_token
        ids = tokenizer(text, truncation=True, max_length=1024)["input_ids"]
        return {"input_ids": ids, "labels": ids, "attention_mask": [1]*len(ids)}
    return _inner

train_tok = train_dataset.map(pretokenize(basic_formatting_function), remove_columns=train_dataset.column_names)
val_tok  = validation_dataset.map(pretokenize(basic_formatting_function), remove_columns=validation_dataset.column_names)

Map: 100%|██████████| 101/101 [00:00<00:00, 157.67 examples/s]


### Initialize Experiment

In [10]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="experkrj_16", mode="fit")

The previously running experiment experkrj_18 was forcibly ended. Created a new experiment 'experkrj_19' with Experiment ID: 20 and Metric Experiment ID: 9 at /home/sjrao/rapidfireai/rapidfire_experiments/experkrj_19


### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [ ]:
import torch

QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
ATTN_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
ALL_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


configs_spec = [

    ("A1_basic_r8", QWEN, basic_formatting_function,  8,   16,   1e-4, 240,  ALL_MODULES),
    ("A2_pkfk_r8", QWEN, pkfk_formatting_function,   8,   16,   1e-4, 240,  ALL_MODULES),
    ("A3_sorted_r8", QWEN, sorted_formatting_function,  8,   16,    1e-4, 240,  ALL_MODULES),

    ("B1_basic_r16",    QWEN, basic_formatting_function,  16,  32,   1e-4, 240,  ALL_MODULES),
    ("B2_basic_r32",    QWEN, basic_formatting_function,  32,  64,   1e-4, 240,  ALL_MODULES),

    ("C1_basic_lr2e4",  QWEN, basic_formatting_function,  8,   16,   2e-4, 240,  ALL_MODULES),
    ("C2_basic_lr5e5",  QWEN, basic_formatting_function,  8,   16,   5e-5, 240,  ALL_MODULES),

    ("D1_basic_attn",   QWEN, basic_formatting_function,  16,  32,   1e-4, 240,  ATTN_MODULES),

    ("E1_pkfk_r16_360", QWEN, pkfk_formatting_function,   16,  32,   1e-4, 360,  ALL_MODULES),
]

all_configs = []
for label, model_name, fmt_func, r, alpha, lr, max_steps, target_modules in configs_spec:
    model_kwargs = {
        "torch_dtype": torch.float16,
        "use_cache": False,
    }
    all_configs.append(RFModelConfig(
        model_name=model_name,
        peft_config=RFLoraConfig(
            r=r,
            lora_alpha=alpha,
            lora_dropout=0.1,
            target_modules=target_modules,
            bias="none",
        ),
        training_args=RFSFTConfig(
        learning_rate=lr,
        lr_scheduler_type="linear",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        max_steps=max_steps,
        gradient_accumulation_steps=4,
        gradient_checkpointing=False,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=20,
        bf16=False,
        fp16=True,
        ),
        model_type="causal_lm",
        model_kwargs=model_kwargs,
        formatting_func=None,
    ))

print(f"Total configs: {len(all_configs)}")
for i, (label, *_) in enumerate(configs_spec):
    print(f"  [{i+1}] {label}")
config_set = List(all_configs)

Total configs: 3
  [1] A1_basic_r8
  [2] A2_pkfk_r8
  [3] A3_sorted_r8


#### Define Model Creation Function for All Model Types Across Configs

In [12]:
def sample_create_model(model_config):
    """Creates model + tokenizer with conservative CUDA settings for worker stability."""
    import gc
    import os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_name = model_config["model_name"]
    model_kwargs = dict(model_config["model_kwargs"])
    model_kwargs.pop("device_map", None)
    model_kwargs.setdefault("low_cpu_mem_usage", True)

    use_4bit = model_kwargs.pop("use_4bit", False)
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model_kwargs["quantization_config"] = bnb_config

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return (model, tokenizer)

#### Generate Config Group

In [13]:
config_group = RFGridSearch(
    configs=config_set,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [14]:
train_mapped = train_dataset.map(basic_formatting_function)
val_mapped = validation_dataset.map(basic_formatting_function)

Map: 100%|██████████| 101/101 [00:00<00:00, 892.99 examples/s]


In [ ]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_tok,
    val_tok,
    num_chunks=1,
    seed=42,
)

/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


INFO 05-30 21:27:47 [__init__.py:216] Automatically detected platform cuda.


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_generation.py:41: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


Started 1 worker processes successfully
Created workers


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_generation.py:41: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


### End Current Experiment

In [9]:
# experiment.end()
Experiment(experiment_name="experkrj_16", mode="fit").end()

Experiment experkrj_16 is currently running. Returning the same experiment object.
Experiment experkrj_16 ended
Workers stopped
